# Structured scorer: no-grouping all-24 evaluation

This is an evaluation-only ablation of the completed five-fold scorer. It reuses the original 9,535-row frozen feature cache and each fold's `best.pt`. It does not load augmented data, build grouping candidates, use event splits, or apply top-8 pruning. The full-sequence verifier directly scores all 24 permutations.

This does **not** retrain a grouping-free model. It measures how the already-trained verifier behaves when grouping is removed at validation time.

## 1. Paths and GPU policy

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path(os.environ.get("SNU_DRIVE_ROOT", "/content/drive/MyDrive/snu ai"))
BUNDLE_PATH = DRIVE_ROOT / "snu_colab_full_train_bundle.tar"
CODE_BUNDLE_PATH = DRIVE_ROOT / "snu_colab_resident_training_code.tar"
WORKDIR = Path(os.environ.get("SNU_WORKDIR", "/content/snu_ai"))
GROUP_FOLDS_CSV = WORKDIR / "snuaichallenge_data/splits/group_stratified_folds_5.csv"
DRIVE_GROUP_FOLDS_CSV = DRIVE_ROOT / "splits/group_stratified_folds_5.csv"
DRIVE_CACHE_SHARDS = DRIVE_ROOT / "cache/full_train_9535_shards"
CHECKPOINT_ROOT = DRIVE_ROOT / "results/full_24class_group_stratified_v1"
OUTPUT_ROOT = DRIVE_ROOT / "results/no_grouping_all24_group_stratified_eval_v1"
LOCAL_CACHE_ROOT = WORKDIR / "feature_cache/full_train_9535"

CACHE_LOAD_BATCH_SIZE = 256
MIN_EVAL_BATCH_SIZE = 128
MAX_EVAL_BATCH_SIZE = 4096
GPU_MEMORY_UTILIZATION_LIMIT = 0.90

assert CACHE_LOAD_BATCH_SIZE > 0
assert 1 <= MIN_EVAL_BATCH_SIZE <= MAX_EVAL_BATCH_SIZE
assert 0.5 <= GPU_MEMORY_UTILIZATION_LIMIT < 1.0

## 2. Mount Drive, refresh code, and validate the MD5 group-stratified split

In [ ]:
import importlib
import json
import subprocess
import sys
from collections import Counter

import torch

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
else:
    print("google.colab is unavailable; configured paths will be used directly.")

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")
if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"Training bundle not found: {BUNDLE_PATH}")
if not CODE_BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"Code bundle not found: {CODE_BUNDLE_PATH}")

WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_CSV = WORKDIR / "snuaichallenge_data/train.csv"
IMAGE_ROOT = WORKDIR / "snuaichallenge_data/train"
if not DRIVE_GROUP_FOLDS_CSV.is_file():
    if not TRAIN_CSV.is_file() or not IMAGE_ROOT.is_dir():
        subprocess.run(["tar", "-xf", str(BUNDLE_PATH), "-C", str(WORKDIR)], check=True)
elif not TRAIN_CSV.is_file():
    subprocess.run(
        ["tar", "-xf", str(BUNDLE_PATH), "-C", str(WORKDIR),
         "snuaichallenge_data/train.csv"],
        check=True,
    )
subprocess.run(
    ["tar", "-xf", str(CODE_BUNDLE_PATH), "-C", str(WORKDIR)],
    check=True,
)
for module_name in list(sys.modules):
    if module_name == "snu_ordering" or module_name.startswith("snu_ordering."):
        del sys.modules[module_name]
importlib.invalidate_caches()
sys.path.insert(0, str(WORKDIR))

import ast
import hashlib
import json
import shutil
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import networkx as nx
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold

TRAIN_CSV = WORKDIR / "snuaichallenge_data/train.csv"
IMAGE_ROOT = WORKDIR / "snuaichallenge_data/train"
GROUP_SPLIT_SEED = 42
GROUP_SPLIT_FOLDS = 5
MAX_SAMPLES_PER_HASH = 3
GROUP_SPLIT_METHOD = "md5_stratified_group_kfold_v1"


def _compute_md5(path):
    with path.open("rb") as file:
        return hashlib.md5(file.read()).hexdigest()


def _build_md5_group_stratified_folds(train_csv, image_root):
    train = pd.read_csv(train_csv)
    train["answer_tuple"] = train["Answer"].map(
        lambda value: str(ast.literal_eval(value))
    )

    image_records = []
    for _, row in train.iterrows():
        for image_index in range(1, 5):
            image_records.append(
                {
                    "sample_id": row["Id"],
                    "path": image_root / str(row["Id"]) / str(row[f"Input_{image_index}"]),
                }
            )

    print(f"Computing MD5 for {len(image_records)} training images...")
    with ThreadPoolExecutor(max_workers=32) as executor:
        hashes = list(executor.map(_compute_md5, [record["path"] for record in image_records]))
    for record, image_hash in zip(image_records, hashes, strict=True):
        record["hash"] = image_hash

    hash_counter = defaultdict(set)
    for record in image_records:
        hash_counter[record["hash"]].add(record["sample_id"])
    hash_to_samples = {
        image_hash: sample_ids
        for image_hash, sample_ids in hash_counter.items()
        if len(sample_ids) <= MAX_SAMPLES_PER_HASH
    }

    graph = nx.Graph()
    graph.add_nodes_from(train["Id"].unique())
    for sample_ids in hash_to_samples.values():
        sample_ids = list(sample_ids)
        for left in range(len(sample_ids)):
            for right in range(left + 1, len(sample_ids)):
                graph.add_edge(sample_ids[left], sample_ids[right])

    components = list(nx.connected_components(graph))
    id_to_cluster = {
        sample_id: cluster_id
        for cluster_id, component in enumerate(components)
        for sample_id in component
    }
    train["cluster_id"] = train["Id"].map(id_to_cluster)

    splitter = StratifiedGroupKFold(
        n_splits=GROUP_SPLIT_FOLDS,
        shuffle=True,
        random_state=GROUP_SPLIT_SEED,
    )
    train["Fold"] = -1
    for fold, (_, validation_indices) in enumerate(
        splitter.split(
            train,
            train["answer_tuple"],
            groups=train["cluster_id"],
        )
    ):
        train.loc[validation_indices, "Fold"] = fold

    train["split_method"] = GROUP_SPLIT_METHOD
    train["split_seed"] = GROUP_SPLIT_SEED
    train["split_max_samples_per_hash"] = MAX_SAMPLES_PER_HASH
    train["split_sklearn_version"] = sklearn.__version__
    build_summary = {
        "clusters": len(components),
        "excluded_frequent_hashes": sum(
            len(sample_ids) > MAX_SAMPLES_PER_HASH
            for sample_ids in hash_counter.values()
        ),
    }
    return train, build_summary


def _validate_md5_group_stratified_folds(folded, raw_train):
    required = {
        "Id", "Answer", "No_ordering", "answer_tuple", "cluster_id", "Fold",
        "split_method", "split_seed", "split_max_samples_per_hash",
        "split_sklearn_version",
    }
    missing = required.difference(folded.columns)
    if missing:
        raise ValueError(f"Group split is missing columns: {sorted(missing)}")
    if folded["Id"].astype(str).tolist() != raw_train["Id"].astype(str).tolist():
        raise ValueError("Group split IDs/order do not match train.csv")
    if folded["Id"].duplicated().any():
        raise ValueError("Group split contains duplicate IDs")

    expected_answers = raw_train["Answer"].map(lambda value: str(ast.literal_eval(value)))
    if folded["answer_tuple"].astype(str).tolist() != expected_answers.tolist():
        raise ValueError("Group split Answer labels do not match train.csv")
    if set(folded["split_method"].astype(str)) != {GROUP_SPLIT_METHOD}:
        raise ValueError("Unexpected group split method")
    if set(folded["split_seed"].astype(int)) != {GROUP_SPLIT_SEED}:
        raise ValueError("Unexpected group split seed")
    if set(folded["split_max_samples_per_hash"].astype(int)) != {MAX_SAMPLES_PER_HASH}:
        raise ValueError("Unexpected MD5 reuse threshold")

    folded["Fold"] = folded["Fold"].astype(int)
    if set(folded["Fold"]) != set(range(GROUP_SPLIT_FOLDS)):
        raise ValueError("Expected folds 0 through 4")
    cross_fold_clusters = int(
        (folded.groupby("cluster_id")["Fold"].nunique() > 1).sum()
    )
    if cross_fold_clusters:
        raise ValueError(f"{cross_fold_clusters} MD5 clusters cross folds")

    identity_mask = folded["answer_tuple"].eq("[1, 2, 3, 4]")
    no_ordering_mask = folded["No_ordering"].astype(str).str.lower().eq("true")
    if not identity_mask.equals(no_ordering_mask):
        raise ValueError("Identity labels and No_ordering=True rows differ")
    classes_per_fold = folded.groupby("Fold")["answer_tuple"].nunique().to_dict()
    if set(classes_per_fold.values()) != {24}:
        raise ValueError(f"Every fold must contain all 24 classes: {classes_per_fold}")

    return {
        "method": GROUP_SPLIT_METHOD,
        "rows": len(folded),
        "classes": folded["answer_tuple"].nunique(),
        "identity_rows": int(identity_mask.sum()),
        "clusters": folded["cluster_id"].nunique(),
        "cross_fold_clusters": cross_fold_clusters,
        "fold_counts": {
            str(fold): int(count)
            for fold, count in folded["Fold"].value_counts().sort_index().items()
        },
        "built_with_sklearn": str(folded["split_sklearn_version"].iloc[0]),
    }


GROUP_FOLDS_CSV.parent.mkdir(parents=True, exist_ok=True)
DRIVE_GROUP_FOLDS_CSV.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_GROUP_FOLDS_CSV.is_file():
    shutil.copy2(DRIVE_GROUP_FOLDS_CSV, GROUP_FOLDS_CSV)
elif not GROUP_FOLDS_CSV.is_file():
    folded_frame, group_split_build = _build_md5_group_stratified_folds(
        TRAIN_CSV,
        IMAGE_ROOT,
    )
    temporary_csv = GROUP_FOLDS_CSV.with_suffix(".csv.tmp")
    folded_frame.to_csv(temporary_csv, index=False)
    temporary_csv.replace(GROUP_FOLDS_CSV)
    print("group split build:", json.dumps(group_split_build, indent=2))

raw_train_frame = pd.read_csv(TRAIN_CSV)
group_fold_frame = pd.read_csv(GROUP_FOLDS_CSV)
group_split_summary = _validate_md5_group_stratified_folds(
    group_fold_frame,
    raw_train_frame,
)
if not DRIVE_GROUP_FOLDS_CSV.is_file():
    temporary_drive_csv = DRIVE_GROUP_FOLDS_CSV.with_suffix(".csv.tmp")
    shutil.copy2(GROUP_FOLDS_CSV, temporary_drive_csv)
    temporary_drive_csv.replace(DRIVE_GROUP_FOLDS_CSV)
print("group split:", json.dumps(group_split_summary, indent=2))

from snu_ordering.data import load_samples

samples = load_samples(
    GROUP_FOLDS_CSV,
    WORKDIR / "snuaichallenge_data/train",
    require_labels=True,
    validate_images=False,
)
answer_counts = Counter(sample.rank_label for sample in samples)
fold_counts = Counter(sample.fold for sample in samples)
identity_count = answer_counts[(1, 2, 3, 4)]
assert len(samples) == 9535
assert len(answer_counts) == 24
assert identity_count == 1478
assert all(sample.fold in range(5) for sample in samples)
assert all(sample.permutation is not None for sample in samples)
for fold in range(5):
    if not (CHECKPOINT_ROOT / f"fold_{fold}/best.pt").is_file():
        raise FileNotFoundError(f"Missing fold checkpoint: {CHECKPOINT_ROOT / f'fold_{fold}/best.pt'}")

torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
DEVICE = torch.device("cuda")
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))
print("rows / classes / identity:", len(samples), len(answer_counts), identity_count)
print("fold counts:", dict(sorted(fold_counts.items())))

## 3. Restore the existing feature cache with progress and ETA

In [ ]:
import tarfile
import time

from snu_ordering.qwen_representations import cache_path_for

def format_duration(seconds):
    if seconds is None or seconds == float("inf"):
        return "--:--:--"
    seconds = max(0, int(round(seconds)))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def print_progress(prefix, completed, total, started_at, extra="", unit="samples"):
    elapsed = time.monotonic() - started_at
    rate = completed / elapsed if elapsed > 0 and completed > 0 else 0.0
    remaining = total - completed
    eta = remaining / rate if rate > 0 else float("inf")
    percent = 100.0 * completed / total if total else 100.0
    suffix = f" | {extra}" if extra else ""
    print(
        f"{prefix} {completed}/{total} ({percent:6.2f}%) | "
        f"remaining={remaining} | elapsed={format_duration(elapsed)} | "
        f"ETA={format_duration(eta)} | {rate:.1f} {unit}/s{suffix}",
        flush=True,
    )

LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
cache_shards = sorted(DRIVE_CACHE_SHARDS.glob("cache_*.tar"))
if not cache_shards:
    raise FileNotFoundError(f"No feature-cache shards found: {DRIVE_CACHE_SHARDS}")
restore_started = time.monotonic()
for shard_index, shard_path in enumerate(cache_shards, 1):
    with tarfile.open(shard_path, "r") as archive:
        archive.extractall(LOCAL_CACHE_ROOT)
    print_progress(
        "restore shards", shard_index, len(cache_shards), restore_started,
        extra=shard_path.name, unit="shards",
    )
missing_cache = [
    sample.sample_id for sample in samples
    if not cache_path_for(LOCAL_CACHE_ROOT, sample.sample_id).is_file()
]
if missing_cache:
    raise FileNotFoundError(
        f"Missing {len(missing_cache)} cached representations; first IDs: {missing_cache[:5]}"
    )
print(f"cache ready: {len(samples)}/{len(samples)} original samples")

## 4. Load only the needed frozen features into VRAM

The grouping path needs event features and candidate plans. This ablation omits both and keeps only ConvNeXt frame features, Qwen frame features, and full-caption features resident on the GPU.

In [ ]:
import gc

from snu_ordering.permutations import ALL_PERMUTATIONS

first_payload = torch.load(
    cache_path_for(LOCAL_CACHE_ROOT, samples[0].sample_id),
    map_location="cpu",
    weights_only=True,
)
convnext_shape = tuple(first_payload["convnext_frame_features"].shape)
qwen_shape = tuple(first_payload["qwen_frame_features"].shape)
caption_shape = tuple(first_payload["full_caption_feature"].shape)
assert convnext_shape[0] == qwen_shape[0] == 4
convnext_features = torch.empty(
    (len(samples), *convnext_shape), dtype=torch.float16, device=DEVICE
)
qwen_features = torch.empty(
    (len(samples), *qwen_shape), dtype=torch.float16, device=DEVICE
)
caption_features = torch.empty(
    (len(samples), *caption_shape), dtype=torch.float16, device=DEVICE
)
del first_payload

load_started = time.monotonic()
for start in range(0, len(samples), CACHE_LOAD_BATCH_SIZE):
    end = min(start + CACHE_LOAD_BATCH_SIZE, len(samples))
    convnext_chunk = []
    qwen_chunk = []
    caption_chunk = []
    for sample in samples[start:end]:
        payload = torch.load(
            cache_path_for(LOCAL_CACHE_ROOT, sample.sample_id),
            map_location="cpu",
            weights_only=True,
        )
        if payload.get("cache_schema") != 1 or payload["sample_id"] != sample.sample_id:
            raise ValueError(f"Invalid cache payload for {sample.sample_id}")
        convnext_chunk.append(payload["convnext_frame_features"].half())
        qwen_chunk.append(payload["qwen_frame_features"].half())
        caption_chunk.append(payload["full_caption_feature"].half())
    convnext_features[start:end].copy_(torch.stack(convnext_chunk).to(DEVICE))
    qwen_features[start:end].copy_(torch.stack(qwen_chunk).to(DEVICE))
    caption_features[start:end].copy_(torch.stack(caption_chunk).to(DEVICE))
    del convnext_chunk, qwen_chunk, caption_chunk, payload
    print_progress("VRAM feature load", end, len(samples), load_started)

permutation_to_index = {
    permutation: index for index, permutation in enumerate(ALL_PERMUTATIONS)
}
targets = torch.tensor(
    [permutation_to_index[sample.permutation] for sample in samples], dtype=torch.long
)
fold_ids = torch.tensor([sample.fold for sample in samples], dtype=torch.long)
no_ordering_mask = torch.tensor([sample.no_ordering is True for sample in samples])
resident_bytes = sum(
    tensor.numel() * tensor.element_size()
    for tensor in (convnext_features, qwen_features, caption_features)
)
print(f"GPU-resident no-grouping features: {resident_bytes / 1024**3:.3f} GiB")
print(f"CUDA allocated: {torch.cuda.memory_allocated() / 1024**3:.3f} GiB")
gc.collect()

## 5. Define all-24 scoring and automatic batch tuning

In [ ]:
from snu_ordering.permutations import ALL_PERMUTATIONS, perm_to_rank_label
from snu_ordering.structured_scorer import LightweightStructuredScorer

PERMUTATION_TENSOR = torch.tensor(ALL_PERMUTATIONS, device=DEVICE, dtype=torch.long)
AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

def load_fold_scorer(fold):
    checkpoint_path = CHECKPOINT_ROOT / f"fold_{fold}/best.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    scorer_config = checkpoint.get("scorer_config", {})
    scorer = LightweightStructuredScorer(
        hidden_dim=int(scorer_config.get("hidden_dim", 256)),
        verifier=scorer_config.get("verifier", "transformer"),
    ).to(DEVICE)
    scorer.load_state_dict(checkpoint["model_state_dict"])
    scorer.eval().requires_grad_(False)
    return scorer, checkpoint_path

def score_all24_without_grouping(scorer, sample_indices):
    gpu_indices = sample_indices.to(DEVICE, non_blocking=True)
    raw_frames = torch.cat(
        [convnext_features[gpu_indices], qwen_features[gpu_indices]], dim=-1
    )
    frames = scorer.frame_projection(raw_frames)
    full_caption = scorer.text_projection(caption_features[gpu_indices])
    ordered_frames = frames[:, PERMUTATION_TENSOR].flatten(end_dim=1)
    repeated_caption = full_caption[:, None, :].expand(
        -1, len(ALL_PERMUTATIONS), -1
    ).flatten(end_dim=1)
    return scorer._score_full_sequence_batch(
        ordered_frames, repeated_caption
    ).reshape(len(sample_indices), len(ALL_PERMUTATIONS))

def choose_eval_batch_size(scorer, probe_indices):
    maximum = min(MAX_EVAL_BATCH_SIZE, len(probe_indices))
    candidates = []
    size = min(MIN_EVAL_BATCH_SIZE, maximum)
    while size < maximum:
        candidates.append(size)
        size = min(size * 2, maximum)
    candidates.append(maximum)
    candidates = sorted(set(candidates))
    total_memory = torch.cuda.get_device_properties(0).total_memory
    best = None
    for candidate in candidates:
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            started = time.monotonic()
            with torch.inference_mode(), torch.autocast(
                device_type="cuda", dtype=AMP_DTYPE, enabled=True
            ):
                probe_scores = score_all24_without_grouping(
                    scorer, probe_indices[:candidate]
                )
            torch.cuda.synchronize()
            elapsed = time.monotonic() - started
            peak = torch.cuda.max_memory_allocated()
            fraction = peak / total_memory
            print(
                f"batch probe={candidate} | {elapsed:.3f}s | "
                f"peak={peak / 1024**3:.2f} GiB ({fraction:.1%})",
                flush=True,
            )
            del probe_scores
            if fraction > GPU_MEMORY_UTILIZATION_LIMIT:
                break
            best = candidate
        except torch.OutOfMemoryError:
            print(f"batch probe={candidate} -> CUDA OOM; keeping {best}", flush=True)
            torch.cuda.empty_cache()
            break
    if best is None:
        raise RuntimeError("Even the minimum evaluation batch does not fit in VRAM.")
    print(f"selected evaluation batch size: {best}")
    return best

## 6. Evaluate all five held-out folds

In [ ]:
import csv

def write_json_atomic(path, payload):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as file:
        json.dump(payload, file, ensure_ascii=False, indent=2)
        file.write("\n")
    temporary.replace(path)

def evaluate_fold(fold, scorer, batch_size, global_completed, global_started):
    fold_indices = torch.nonzero(fold_ids == fold, as_tuple=False).squeeze(1)
    fold_targets = targets[fold_indices]
    predictions = []
    fold_started = time.monotonic()
    start = 0
    active_batch_size = batch_size
    torch.cuda.reset_peak_memory_stats()
    while start < len(fold_indices):
        end = min(start + active_batch_size, len(fold_indices))
        batch_indices = fold_indices[start:end]
        try:
            with torch.inference_mode(), torch.autocast(
                device_type="cuda", dtype=AMP_DTYPE, enabled=True
            ):
                scores = score_all24_without_grouping(scorer, batch_indices)
                batch_predictions = scores.argmax(dim=1).cpu()
            predictions.append(batch_predictions)
            del scores, batch_predictions
            start = end
            torch.cuda.synchronize()
            print_progress(
                f"fold {fold}", start, len(fold_indices), fold_started,
                extra=f"batch={active_batch_size}",
            )
            print_progress(
                "all folds", global_completed + start, len(samples), global_started,
                extra=f"fold={fold}",
            )
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            if active_batch_size <= 1:
                raise
            active_batch_size = max(1, active_batch_size // 2)
            print(f"fold {fold}: CUDA OOM, retrying with batch={active_batch_size}", flush=True)

    fold_predictions = torch.cat(predictions)
    correct = fold_predictions == fold_targets
    fold_no_ordering = no_ordering_mask[fold_indices]
    elapsed = time.monotonic() - fold_started
    metrics = {
        "fold": fold,
        "mode": "full_sequence_only_all_24_no_grouping",
        "samples": len(fold_indices),
        "correct": int(correct.sum()),
        "exact_match_accuracy": float(correct.float().mean()),
        "no_ordering_samples": int(fold_no_ordering.sum()),
        "no_ordering_correct": int((correct & fold_no_ordering).sum()),
        "ordering_samples": int((~fold_no_ordering).sum()),
        "ordering_correct": int((correct & ~fold_no_ordering).sum()),
        "elapsed_seconds": elapsed,
        "samples_per_second": len(fold_indices) / elapsed,
        "requested_batch_size": batch_size,
        "final_batch_size": active_batch_size,
        "peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
    }
    fold_dir = OUTPUT_ROOT / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    prediction_tmp = fold_dir / "predictions.csv.tmp"
    prediction_path = fold_dir / "predictions.csv"
    with prediction_tmp.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(
            file, fieldnames=["Id", "Fold", "Target", "Prediction", "Correct"]
        )
        writer.writeheader()
        for local_index, prediction_index in enumerate(fold_predictions.tolist()):
            sample_index = int(fold_indices[local_index])
            writer.writerow({
                "Id": samples[sample_index].sample_id,
                "Fold": fold,
                "Target": json.dumps(samples[sample_index].rank_label),
                "Prediction": json.dumps(perm_to_rank_label(ALL_PERMUTATIONS[prediction_index])),
                "Correct": bool(correct[local_index]),
            })
    prediction_tmp.replace(prediction_path)
    write_json_atomic(fold_dir / "metrics.json", metrics)
    print(json.dumps(metrics, indent=2), flush=True)
    return metrics

largest_fold = max(range(5), key=lambda fold: fold_counts[fold])
probe_indices = torch.nonzero(fold_ids == largest_fold, as_tuple=False).squeeze(1)
probe_scorer, _ = load_fold_scorer(largest_fold)
EVAL_BATCH_SIZE = choose_eval_batch_size(probe_scorer, probe_indices)
del probe_scorer
gc.collect()
torch.cuda.empty_cache()

fold_metrics = []
global_started = time.monotonic()
global_completed = 0
for fold in range(5):
    scorer, checkpoint_path = load_fold_scorer(fold)
    print(f"fold {fold}: checkpoint={checkpoint_path}", flush=True)
    metrics = evaluate_fold(
        fold, scorer, EVAL_BATCH_SIZE, global_completed, global_started
    )
    fold_metrics.append(metrics)
    global_completed += metrics["samples"]
    del scorer
    gc.collect()
    torch.cuda.empty_cache()

## 7. Aggregate and persist the ablation result

In [ ]:
total_samples = sum(row["samples"] for row in fold_metrics)
total_correct = sum(row["correct"] for row in fold_metrics)
no_ordering_samples = sum(row["no_ordering_samples"] for row in fold_metrics)
no_ordering_correct = sum(row["no_ordering_correct"] for row in fold_metrics)
ordering_samples = sum(row["ordering_samples"] for row in fold_metrics)
ordering_correct = sum(row["ordering_correct"] for row in fold_metrics)
summary = {
    "mode": "full_sequence_only_all_24_no_grouping",
    "description": (
        "Existing fold checkpoints; original frozen features only; no grouping, "
        "event splits, candidate plans, visual prior, or top-M pruning."
    ),
    "folds": fold_metrics,
    "samples": total_samples,
    "correct": total_correct,
    "weighted_exact_match_accuracy": total_correct / total_samples,
    "no_ordering_samples": no_ordering_samples,
    "no_ordering_exact_match_accuracy": no_ordering_correct / no_ordering_samples,
    "ordering_samples": ordering_samples,
    "ordering_exact_match_accuracy": ordering_correct / ordering_samples,
    "evaluation_batch_size": EVAL_BATCH_SIZE,
    "feature_source": "cache/full_train_9535_shards",
    "checkpoint_source": "results/full_24class_group_stratified_v1/fold_*/best.pt",
}
baseline_path = CHECKPOINT_ROOT / "five_fold_summary.json"
if baseline_path.is_file():
    with baseline_path.open("r", encoding="utf-8") as file:
        baseline = json.load(file)
    baseline_accuracy = baseline.get("weighted_exact_match_accuracy")
    if baseline_accuracy is not None:
        summary["grouping_top8_baseline_exact_match_accuracy"] = baseline_accuracy
        summary["delta_vs_grouping_top8"] = (
            summary["weighted_exact_match_accuracy"] - baseline_accuracy
        )
assert summary["samples"] == 9535
write_json_atomic(OUTPUT_ROOT / "five_fold_summary.json", summary)
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 8. Interpretation boundary

Compare `weighted_exact_match_accuracy` with the new MD5 group-stratified baseline at `results/full_24class_group_stratified_v1/five_fold_summary.json`; legacy EXP-003 used different folds. A gain means the top-8/grouping decision was suppressing useful verifier choices. A loss means the jointly trained verifier still depends on the grouping score. Either result is only an inference-time ablation; training a genuinely grouping-free scorer would be a separate experiment and requires GPU configuration confirmation before training.